In [4]:
# Install PyCryptodome if not already installed
!pip install pycryptodome

# Import necessary libraries
from Crypto.Hash import CMAC
from Crypto.Cipher import AES
import binascii
from IPython.display import display, HTML
import os


offset = 0x4000
length = 0x2000

# Helper Functions
def bytes_to_hex(byte_data):
    return binascii.hexlify(byte_data).decode('utf-8').upper()

def compute_aes_cmac(key, data):
    """
    Computes AES-CMAC for the given key and data.
    
    :param key: AES key (16, 24, or 32 bytes)
    :param data: Data to compute CMAC over (bytes)
    :return: Calculated CMAC (bytes)
    """
    cobj = CMAC.new(key, ciphermod=AES)
    cobj.update(data)
    return cobj.digest()

def extract_data_segment(data, offset, length):
    """
    Extracts a segment from the binary data based on offset and length.
    
    :param data: The complete binary data (bytes)
    :param offset: The starting point (integer)
    :param length: The number of bytes to extract (integer)
    :return: Extracted data segment (bytes)
    """
    end = offset + length
    if offset < 0 or end > len(data):
        raise ValueError("Offset and length are out of bounds.")
    return data[offset:end]

def display_steps(aes_key_hex, offset, length, binary_hex, data_segment_hex, calculated_cmac_hex, received_cmac_hex=None):
    html_content = f"""
    <h2>AES-CMAC Calculation Steps</h2>
    <ol>
        <li>
            <strong>AES Key:</strong>
            <pre>{aes_key_hex}</pre>
        </li>
        <li>
            <strong>Data Segment for CMAC:</strong>
            <ul>
                <li><strong>Offset:</strong> {offset}</li>
                <li><strong>Length:</strong> {length} bytes</li>
                <li><strong>Data Segment (Hex):</strong>
                    <pre>{data_segment_hex}</pre>
                </li>
            </ul>
        </li>
        <li>
            <strong>Compute AES-CMAC:</strong>
            <ul>
                <li>Initialize CMAC with AES cipher.</li>
                <li>Update CMAC with data segment.</li>
                <li>Finalize and obtain the CMAC digest.</li>
            </ul>
        </li>
        <li>
            <strong>Calculated AES-CMAC:</strong>
            <pre>{calculated_cmac_hex}</pre>
        </li>
    </ol>
    """
    
    if received_cmac_hex:
        # Compare the CMACs
        is_match = calculated_cmac_hex == received_cmac_hex.upper()
        compare_result = "CMACs são iguais ✅" if is_match else "CMACs são diferentes ❌"
        
        html_content += f"""
        <h2>Comparison with Received CMAC</h2>
        <ol>
            <li>
                <strong>Received AES-CMAC:</strong>
                <pre>{received_cmac_hex}</pre>
            </li>
            <li>
                <strong>Comparison Result:</strong>
                <pre>{compare_result}</pre>
            </li>
        </ol>
        """
    
    display(HTML(html_content))

# --------------------------
# 1. Read the Binary File
# --------------------------

# Path to your binary file
binary_file_path = 'fcx_msp_v3.0.1.3.bin'

# Ensure the file exists
if not os.path.exists(binary_file_path):
    # For demonstration, let's create a sample binary file
    with open(binary_file_path, 'wb') as f:
        f.write(os.urandom(128))  # Write 128 random bytes

# Read the binary file
with open(binary_file_path, 'rb') as f:
    binary_data = f.read()

# Convert binary data to hex
binary_hex = bytes_to_hex(binary_data)

# Extract the data segment
try:
    data_segment = extract_data_segment(binary_data, offset, length)
    data_segment_hex = bytes_to_hex(data_segment)
    display(HTML(f"<h3>Data Segment (Hex):</h3><pre>Offset: {offset}, Length: {length}\n{data_segment_hex}</pre>"))
except ValueError as ve:
    display(HTML(f"<h3 style='color:red;'>Error:</h3><pre>{ve}</pre>"))

# --------------------------
# 3. Define the AES Key
# --------------------------

# Example AES-128 key (16 bytes)
aes_key_hex = 'FFFFAAAAFFFFAAAAFFFFAAAAFFFFAAAA'  # Replace with your key
aes_key = binascii.unhexlify(aes_key_hex)

# Display the AES Key
display(HTML(f"<h3>AES Key (Hex):</h3><pre>{aes_key_hex}</pre>"))

# --------------------------
# 4. Compute AES-CMAC
# --------------------------

# Compute AES-CMAC
calculated_cmac = compute_aes_cmac(aes_key, data_segment)
calculated_cmac_hex = bytes_to_hex(calculated_cmac)

# Display the Calculated CMAC
display(HTML(f"<h3>Calculated AES-CMAC (Hex):</h3><pre>{calculated_cmac_hex}</pre>"))

# --------------------------
# 5. Compare with Received CMAC
# --------------------------

# Example received CMAC (replace with actual received CMAC)
received_cmac_hex = '10CC07235247164529B4CB438D87ED63'  # Replace with your received CMAC

# Display the Received CMAC
display(HTML(f"<h3>Received AES-CMAC (Hex):</h3><pre>{received_cmac_hex}</pre>"))

# Display the steps with comparison
display_steps(aes_key_hex, offset, length, binary_hex, data_segment_hex, calculated_cmac_hex, received_cmac_hex)


In [25]:
# Instalar Cryptography se ainda não estiver instalado
# pip install cryptography

from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import cmac
import binascii
import os

# Parâmetros
offset = 0x4000
comprimento = 0x2000
caminho_arquivo_binario = 'fcx_msp_v3.0.1.3.bin'
chave_aes_hex = 'FFFFAAAAFFFFAAAAFFFFAAAAFFFFAAAA'
cmac_recebido_hex = '10CC07235247164529B4CB438D87ED63'

# Funções Auxiliares
def bytes_para_hex(dados_bytes):
    return binascii.hexlify(dados_bytes).decode('utf-8').upper()

def calcular_cmac_aes(chave, dados):
    c = cmac.CMAC(algorithms.AES(chave))
    c.update(dados)
    return c.finalize()

# Ler arquivo binário
with open(caminho_arquivo_binario, 'rb') as f:
    dados_binarios = f.read()

# Extrair segmento de dados
segmento_dados = dados_binarios[offset:offset + comprimento]
if len(segmento_dados) != comprimento:
    raise ValueError("Offset ou comprimento inválido.")

# Converter chave AES e calcular CMAC
chave_aes = binascii.unhexlify(chave_aes_hex)
cmac_calculado = calcular_cmac_aes(chave_aes, segmento_dados)
cmac_calculado_hex = bytes_para_hex(cmac_calculado)

# Imprimir resultados
print(f"CMAC Calculado: {cmac_calculado_hex}")
print(f"CMAC Recebido:   {cmac_recebido_hex}")
print(f"Correspondência:  {'✅' if cmac_calculado_hex == cmac_recebido_hex.upper() else '❌'}")


CMAC Calculado: 10CC07235247164529B4CB438D87ED63
CMAC Recebido:   10CC07235247164529B4CB438D87ED63
Correspondência:  ✅


In [ ]:
def compute_aes_cmac(key, data):
    """
    Computes AES-CMAC for the given key and data.
    
    :param key: AES key (16, 24, or 32 bytes)
    :param data: Data to compute CMAC over (bytes)
    :return: Calculated CMAC (bytes)
    """
    cobj = CMAC.new(key, ciphermod=AES)
    cobj.update(data)
    return cobj.digest()

In [8]:
!python.exe -m pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 621.2 kB/s eta 0:00:03
   ----------------- ---------------------- 0.8/1.8 MB 860.9 kB/s eta 0:00:02
   ----------------------- ---------------- 1.0/1.8 MB 1.0 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 1.2 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 1.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2


In [ ]:
# Install PyCryptodome if not already installed
!pip install pycryptodome

# Import necessary libraries
from Crypto.Hash import CMAC
from Crypto.Cipher import AES
import binascii
from IPython.display import display, HTML

# Constants
BLOCK_SIZE = 16
CONST_RB = 0x87

# Helper Functions
def bytes_to_hex(byte_data):
    return binascii.hexlify(byte_data).decode('utf-8').upper()

def left_shift_one_bit(input_bytes):
    output = bytearray(BLOCK_SIZE)
    overflow = 0
    for i in reversed(range(BLOCK_SIZE)):
        output[i] = ((input_bytes[i] << 1) & 0xFF) | overflow
        overflow = 1 if (input_bytes[i] & 0x80) else 0
    return output

def generate_subkeys(aes_key):
    cipher = AES.new(aes_key, AES.MODE_ECB)
    zero_block = bytes([0] * BLOCK_SIZE)
    L = bytearray(cipher.encrypt(zero_block))
    display_steps("Encrypted Zero Block (L)", bytes_to_hex(L))
    
    # Generate K1
    K1 = left_shift_one_bit(L)
    if L[0] & 0x80:
        K1[-1] ^= CONST_RB
    display_steps("Subkey K1", bytes_to_hex(K1))
    
    # Generate K2
    K2 = left_shift_one_bit(K1)
    if K1[0] & 0x80:
        K2[-1] ^= CONST_RB
    display_steps("Subkey K2", bytes_to_hex(K2))
    
    return K1, K2

def compute_cmac_manual(aes_key, data):
    # Initialize CMAC
    cipher = AES.new(aes_key, AES.MODE_ECB)
    
    # Generate Subkeys
    K1, K2 = generate_subkeys(aes_key)
    
    # Determine if the last block is complete
    n_blocks = len(data) // BLOCK_SIZE
    remaining = len(data) % BLOCK_SIZE
    if remaining == 0 and len(data) != 0:
        complete_block = True
    else:
        complete_block = False
        n_blocks += 1
    
    display_steps("Number of Blocks", str(n_blocks))
    display_steps("Complete Block", str(complete_block))
    
    # Process all blocks except the last
    cmac_prev = bytearray([0] * BLOCK_SIZE)
    for i in range(n_blocks - 1):
        block = bytearray(data[i*BLOCK_SIZE:(i+1)*BLOCK_SIZE])
        display_steps(f"Processing Block {i+1} Before XOR", bytes_to_hex(block))
        
        # XOR with previous CMAC
        for j in range(BLOCK_SIZE):
            block[j] ^= cmac_prev[j]
        display_steps(f"Block {i+1} After XOR with Previous CMAC", bytes_to_hex(block))
        
        # Encrypt the block
        temp_out = bytearray(cipher.encrypt(bytes(block)))
        display_steps(f"Encrypted Block {i+1}", bytes_to_hex(temp_out))
        
        # Update cmac_prev
        cmac_prev = temp_out
    
    # Handle the last block
    if complete_block and len(data) != 0:
        M_last = bytearray(data[(n_blocks-1)*BLOCK_SIZE:(n_blocks)*BLOCK_SIZE])
        display_steps("Last Block is complete. XOR with K1.", "")
        for j in range(BLOCK_SIZE):
            M_last[j] ^= K1[j]
        display_steps("M_last After XOR with K1", bytes_to_hex(M_last))
    else:
        M_last = bytearray([0] * BLOCK_SIZE)
        if len(data) > 0:
            M_last[:remaining] = data[(n_blocks-1)*BLOCK_SIZE:(n_blocks-1)*BLOCK_SIZE + remaining]
        if remaining < BLOCK_SIZE:
            M_last[remaining] = 0x80  # Append '1' bit followed by '0's
        display_steps("Last Block is not complete. Applying padding and XOR with K2.", "")
        display_steps("M_last After Padding", bytes_to_hex(M_last))
        for j in range(BLOCK_SIZE):
            M_last[j] ^= K2[j]
        display_steps("M_last After XOR with K2", bytes_to_hex(M_last))
    
    if len(data) != 0:
        # XOR the last block with the previous CMAC
        for j in range(BLOCK_SIZE):
            M_last[j] ^= cmac_prev[j]
        display_steps("M_last After XOR with Previous CMAC", bytes_to_hex(M_last))
        
        # Encrypt the last block to get the final CMAC
        cmac_out = cipher.encrypt(bytes(M_last))
        display_steps("Final CMAC", bytes_to_hex(cmac_out))
    else:
        # If len(data) is 0, define CMAC as zero block or handle accordingly
        cmac_out = bytes([0] * BLOCK_SIZE)
        display_steps("ul_length is 0. CMAC set to zero block.", "")
        display_steps("CMAC_out", bytes_to_hex(cmac_out))
    
    return cmac_out

def display_steps(label, value):
    html_content = f"""
    <h4>{label}:</h4>
    <pre>{value}</pre>
    """
    display(HTML(html_content))

# Example usage
if __name__ == "__main__":
    # Define the AES key and input data
    aes_key_hex = "FFFFAAAAFFFFAAAAFFFFAAAAFFFFAAAA"
    data_hex = data_segment_hex
    
    # Convert hex to bytes
    aes_key = bytes.fromhex(aes_key_hex)
    data = bytes.fromhex(data_hex)
    
    # Display the AES Key
    display_steps("AES Key (Hex)", aes_key_hex)
    
    # Pad data if necessary (since data is 15 bytes)
    if len(data) < BLOCK_SIZE:
        data_padded = bytearray(data) + bytes([0x80] + [0x00]*(BLOCK_SIZE - len(data) -1))
    else:
        data_padded = data[:BLOCK_SIZE]
    
    # Compute AES-CMAC manually
    calculated_cmac = compute_cmac_manual(aes_key, data)
    
    # Display the Calculated CMAC
    display_steps("Calculated AES-CMAC (Hex)", bytes_to_hex(calculated_cmac))


In [10]:
import sys

def read_binary_section(filename='fcx_msp_v3.0.1.3.bin', offset=0x5000, length=0xF):
    try:
        with open(filename, 'rb') as f:
            f.seek(offset)
            data = f.read(length)
            print(data.hex().upper())
    except FileNotFoundError:
        print(f"File '{filename}' not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    read_binary_section()
    

08008A931A0006248A931000062008
